# 토스증권 Open API — `toss_invest` 테스트 노트북

`toss_invest.py` 모듈의 각 기능을 카테고리별로 점검합니다.

**사전 준비**
1. 이 노트북을 `toss_invest.py`, `requirements.txt` 와 **같은 폴더**에 두세요.
2. 같은 폴더에 `.env` 파일을 만들고 발급받은 키를 넣으세요 (`.env.example` 참고).

```
TOSS_CLIENT_ID=c_xxxxx
TOSS_CLIENT_SECRET=s_xxxxx
TOSS_ACCOUNT_SEQ=        # 선택. 비워두면 아래에서 자동 선택
```

> ⚠️ 별도 sandbox 가 없습니다. **주문(Order) 셀은 실제 자산을 움직입니다.**
> 안전을 위해 주문 셀은 `ENABLE_ORDER_TESTS` 플래그로 잠가 두었고, 소액으로만 테스트하세요.

## 0. 설치 & 임포트

> **설치 오류(PEP 668) 대응**
>
> `externally-managed-environment` 오류가 나면, 시스템 Python이 보호된 상태입니다. 두 가지 중 하나로 해결하세요.
>
> **권장 — 가상환경 커널** (터미널에서):
> ```bash
> python3 -m venv .venv
> source .venv/bin/activate      # Windows: .venv\Scripts\activate
> pip install -r requirements.txt ipykernel
> python -m ipykernel install --user --name toss-test --display-name "Python (toss-test)"
> ```
> 이후 Jupyter 우측 상단에서 커널을 **Python (toss-test)** 로 변경.
>
> **빠른 우회** — 아래 설치 셀에서 `--break-system-packages` 를 추가 (시스템 Python에 직접 설치되니 주의).

In [ ]:
# 패키지 설치
# - 권장: 가상환경(venv/conda) 커널을 만든 뒤 실행하면 그대로 통과합니다.
# - PEP 668 'externally-managed-environment' 오류가 나면 아래 한 줄을 대신 쓰세요(시스템 Python에 직접 설치, 위험 감수):
#     %pip install -q --break-system-packages requests python-dotenv
%pip install -q requests python-dotenv

In [ ]:
import json
from toss_invest import TossInvestClient, TossAPIError, TossAuthError

def show(obj, n=2000):
    """응답을 보기 좋게 출력하는 헬퍼."""
    s = json.dumps(obj, ensure_ascii=False, indent=2, default=str)
    print(s[:n] + ('\n... (생략)' if len(s) > n else ''))

print('임포트 완료')

## 1. 클라이언트 생성 & 인증 확인

`.env` 가 자동 로드됩니다. 토큰은 첫 호출 시 자동 발급·캐싱됩니다.

In [ ]:
toss = TossInvestClient()

# 토큰이 정상 발급되는지 확인 (내부 메서드지만 점검용으로 직접 호출)
token = toss._get_token()
print('토큰 발급 OK:', token[:24], '...')
print('만료(epoch):', toss._token_expiry)

## 2. Market Data — 시세/호가/체결/캔들

인증만 있으면 호출 가능합니다. 예시 종목: 삼성전자 `005930`, 애플 `AAPL`.

In [ ]:
# 현재가 (복수 심볼 가능)
show(toss.get_prices(['005930', 'AAPL']))

In [ ]:
# 호가창
show(toss.get_orderbook('005930'))

In [ ]:
# 최근 체결 내역
show(toss.get_trades('005930', count=10))

In [ ]:
# 상/하한가
show(toss.get_price_limits('005930'))

In [ ]:
# 캔들 (일봉 최근 5개)
show(toss.get_candles('005930', interval='1d', count=5))

## 3. Stock Info — 종목 정보 / 매수 유의사항

In [ ]:
# 종목 기본 정보 (복수 가능)
show(toss.get_stocks(['005930', 'AAPL']))

In [ ]:
# 매수 유의사항
show(toss.get_warnings('005930'))

## 4. Market Info — 환율 / 장 운영 정보

In [ ]:
# USD -> KRW 환율
show(toss.get_exchange_rate('USD', 'KRW'))

In [ ]:
# 국내/해외 장 운영 정보
show(toss.get_market_calendar('KR'))
show(toss.get_market_calendar('US'))

## 5. Account & Asset — 계좌 / 보유 종목

계좌·자산·주문 엔드포인트는 `X-Tossinvest-Account` 헤더(정수)가 필요합니다.
먼저 계좌 목록을 받아 `ACCOUNT` 변수에 설정합니다.

In [ ]:
accounts = toss.get_accounts()
show(accounts)

In [ ]:
# 위 응답을 보고 사용할 계좌 식별자를 지정하세요.
# .env 의 TOSS_ACCOUNT_SEQ 를 설정했다면 None 그대로 두면 됩니다.
ACCOUNT = None   # 예: 12345

if ACCOUNT is None and toss.account_seq is None:
    print('⚠️ ACCOUNT 를 직접 지정하거나 .env 의 TOSS_ACCOUNT_SEQ 를 설정하세요.')
else:
    print('사용 계좌:', ACCOUNT or toss.account_seq)

In [ ]:
# 보유 주식
show(toss.get_holdings(account=ACCOUNT))

## 6. Order Info — 매수가능금액 / 매도가능수량 / 수수료

In [ ]:
# 매수 가능 금액 (currency 필수: 'KRW' | 'USD')
show(toss.get_buying_power('KRW', account=ACCOUNT))

In [ ]:
# 매도 가능 수량
show(toss.get_sellable_quantity('005930', account=ACCOUNT))

In [ ]:
# 매매 수수료
show(toss.get_commissions(account=ACCOUNT))

## 7. Order History — 주문 목록 / 상세

In [ ]:
# 미체결(OPEN) 주문 목록
open_orders = toss.get_orders(status='OPEN', account=ACCOUNT)
show(open_orders)

In [ ]:
# 체결완료(CLOSED) 주문 목록
show(toss.get_orders(status='CLOSED', account=ACCOUNT, limit=10))

In [ ]:
# 특정 주문 상세 (orderId 를 채워서 실행)
ORDER_ID = None  # 예: 'o_xxxxx'
if ORDER_ID:
    show(toss.get_order(ORDER_ID, account=ACCOUNT))
else:
    print('ORDER_ID 를 지정하면 상세를 조회합니다.')

## 8. ⚠️ Order — 주문 생성 / 정정 / 취소 (실거래)

**아래 셀은 실제 주문을 접수합니다.** 의도적으로 실행하려면 `ENABLE_ORDER_TESTS = True` 로 바꾸세요.
전량 실행(Run All) 사고를 막기 위한 안전장치입니다. 반드시 소액(1주)으로 시작하세요.

In [ ]:
ENABLE_ORDER_TESTS = False   # 실제 주문을 내려면 True 로 변경

In [ ]:
# 신규 지정가 매수 (예: 삼성전자 1주, 70,000원)
if ENABLE_ORDER_TESTS:
    res = toss.place_order(
        symbol='005930', side='BUY', order_type='LIMIT',
        quantity=1, price=70000,
        client_order_id='nb-test-001',  # 멱등성 키(10분) — 중복 실행 안전
        account=ACCOUNT,
    )
    show(res)
else:
    print('주문 비활성. ENABLE_ORDER_TESTS = True 로 변경 후 실행하세요.')

In [ ]:
# 주문 정정 (KR 은 quantity 필수, LIMIT 은 price 필수)
MODIFY_ORDER_ID = None
if ENABLE_ORDER_TESTS and MODIFY_ORDER_ID:
    show(toss.modify_order(MODIFY_ORDER_ID, order_type='LIMIT',
                           quantity=1, price=71000, account=ACCOUNT))
else:
    print('정정 비활성 (ENABLE_ORDER_TESTS 와 MODIFY_ORDER_ID 필요).')

In [ ]:
# 주문 취소
CANCEL_ORDER_ID = None
if ENABLE_ORDER_TESTS and CANCEL_ORDER_ID:
    show(toss.cancel_order(CANCEL_ORDER_ID, account=ACCOUNT))
else:
    print('취소 비활성 (ENABLE_ORDER_TESTS 와 CANCEL_ORDER_ID 필요).')

## 9. 에러 처리 예시

`TossAPIError` 는 `status_code`, `code`, `message`, `request_id`, `data` 를 담습니다.

In [ ]:
try:
    # 존재하지 않는 종목 등으로 일부러 에러 유발
    toss.get_orderbook('___INVALID___')
except TossAPIError as e:
    print('status :', e.status_code)
    print('code   :', e.code)
    print('message:', e.message)
    print('reqId  :', e.request_id)
    show(e.data)

In [ ]:
# 세션 정리
toss.close()
print('완료')